In [ ]:
# scraper part 1: collect listing pages (location, size, price, link)

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import math

# set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

# selenium 4.6+ manages chromedriver automatically, no path needed
driver = webdriver.Chrome(options=options)

# function to handle the consent screen
def handle_consent():
    try:
        # wait for the consent button to be clickable
        WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button.cw-btn.cw-btn--lg.cw-btn--green"))
        )
        consent_button = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@data-testid='cw-button-non-targeted-ad']"))
        )
        # force click using JavaScript
        driver.execute_script("arguments[0].click();", consent_button)
        print("Consent handled: Clicked on 'Souhlasím'")

    except Exception as e:
        print("Could not find or interact with the consent screen:", str(e))

# function to scrape a single page using Selenium
def scrape_page(url):
    driver.get(url)

    # wait for the page to load completely
    time.sleep(5)  # adjust sleep duration based on connection speed

    # get the page source after JavaScript has rendered the content
    rendered_page = driver.page_source
    soup = BeautifulSoup(rendered_page, 'html.parser')

    # extract locations, sizes, prices, and links
    locations = soup.select('span.locality.ng-binding')
    sizes = soup.select('span.name.ng-binding')
    prices = soup.select('span.norm-price.ng-binding')
    links = soup.select('a.title')

    page_data = []

    # loop through items and extract data
    for i in range(len(locations)):
        location = locations[i].get_text().strip()
        size = sizes[i].get_text().strip()
        price = prices[i].get_text().replace('\xa0', '').replace('CZK', '').strip()  # clean up price formatting
        link = 'https://www.sreality.cz' + links[i]['href']  # full link

        # append the data to the list
        page_data.append([location, size, price, link])

    return page_data

# function to get total number of pages
def get_total_pages(url):
    driver.get(url)

    # wait for the page to load completely
    time.sleep(5)

    # get the page source and parse it
    rendered_page = driver.page_source
    soup = BeautifulSoup(rendered_page, 'html.parser')

    # find all occurrences of 'span.numero.ng-binding'
    total_listings_tags = soup.find_all('span', class_='numero ng-binding')

    if len(total_listings_tags) > 1:
        # select the second occurrence (index 1)
        total_listings = int(total_listings_tags[1].get_text().replace('\xa0', '').strip())
        print(f"Total listings found: {total_listings}")

        # calculate total pages (20 listings per page)
        listings_per_page = 20
        total_pages = math.ceil(total_listings / listings_per_page)

    else:
        print("Could not find the total number of listings. Defaulting to 50 pages.")
        total_pages = 50  # default fallback

    return total_pages

# main scraping function to iterate through multiple pages dynamically
def scrape_all_pages():
    base_url = "https://www.sreality.cz/en/search/for-sale/apartments/praha"

    # first, handle the consent screen
    driver.get(base_url)

    # ensure the consent screen is handled before proceeding
    handle_consent()

    # then, get the total number of pages
    total_pages = get_total_pages(base_url)
    print(f"Total pages to scrape: {total_pages}")

    all_data = []

    # loop through each page and scrape data
    for page in range(1, total_pages + 1):
        url = f"{base_url}?page={page}"
        print(f"Scraping page {page} of {total_pages}...")
        page_data = scrape_page(url)
        all_data.extend(page_data)  # append page data to main list

        time.sleep(2)  # be polite, avoid rate limiting

    return all_data

# scrape all pages automatically
data = scrape_all_pages()

# close the Selenium browser session
driver.quit()

# convert the data to a DataFrame
df = pd.DataFrame(data, columns=["Location", "Size", "Price (CZK)", "Link"])

# show the DataFrame
print(df)

# optionally save the DataFrame to a CSV file
df.to_csv('apartments_Praha.csv', index=False)


import time
import pandas as pd
import signal
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
import threading

# set up driver (visible mode, selenium 4.6+ handles chromedriver automatically)
def get_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    return webdriver.Chrome(options=options)

# function to handle the consent screen
def handle_consent(driver):
    try:
        WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button.cw-btn.cw-btn--lg.cw-btn--green"))
        )
        consent_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@data-testid='cw-button-non-targeted-ad']"))
        )
        driver.execute_script("arguments[0].click();", consent_button)
        print("Consent handled: Clicked on 'Souhlasím'")
    except Exception:
        pass  # suppress errors if consent screen isn't visible

# function to scrape parameters
def scrape_params(driver):
    try:
        params_section = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.params.clear"))
        )
        groups = params_section.find_elements(By.CSS_SELECTOR, "ul")

        param_dict = {}
        for group in groups:
            items = group.find_elements(By.CSS_SELECTOR, "li")
            for item in items:
                param_label = item.find_element(By.CSS_SELECTOR, "label.param-label").text.strip(':')
                param_value = item.find_element(By.CSS_SELECTOR, "strong.param-value").text.strip()
                param_dict[param_label] = param_value

        return param_dict
    except Exception:
        return {}  # empty dict if parameters section not found

# function to scrape a single page using an existing driver
def scrape_single_page(driver, link):
    driver.get(link)

    # handle consent screen
    handle_consent(driver)

    # scrape parameters
    params = scrape_params(driver)

    # add link to params so we can merge it later
    params['Link'] = link
    return params

# function to log progress every 100 flats scraped
def log_progress(scraped_count):
    if scraped_count % 100 == 0:
        logging.info(f"{scraped_count} flats scraped.")

# global variable to track browser windows and scraped count
drivers = []
scraped_count = 0
count_lock = threading.Lock()  # thread-safe counter for parallel workers

# graceful shutdown function to close all browser windows
def graceful_shutdown(signum, frame):
    print("\nGraceful shutdown initiated...")
    for driver in drivers:
        driver.quit()
    print("All browser windows closed.")
    if signum is not None:
        exit(0)  # only exit on real signal

# register the signal handler
signal.signal(signal.SIGINT, graceful_shutdown)

# function to scrape flats using fixed drivers (browser windows)
def scrape_pages_with_fixed_drivers(df, num_workers=5):
    global drivers, scraped_count
    # initialize the drivers (one for each worker)
    drivers = [get_driver() for _ in range(num_workers)]

    # handle consent on each driver manually the first time they open
    for driver in drivers:
        link = df.iloc[0]['Link']  # triggers consent screen
        driver.get(link)
        handle_consent(driver)

    all_params = []

    # function to assign flats to a specific driver
    def scrape_for_driver(driver, links):
        global scraped_count  # needs global since it's modified in nested scope
        local_params = []
        for link in links:
            try:
                params = scrape_single_page(driver, link)
                local_params.append(params)

                # update scraped count in a thread-safe way
                with count_lock:
                    scraped_count += 1
                    log_progress(scraped_count)
            except Exception as e:
                logging.error(f"Error scraping {link}: {e}")
        return local_params

    # split the DataFrame into chunks for each driver
    chunks = [df[i::num_workers] for i in range(num_workers)]

    # use ThreadPoolExecutor to assign chunks to each driver
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(scrape_for_driver, drivers[i], chunks[i]['Link']) for i in range(num_workers)]

        # collect results as each driver finishes its chunk
        for future in as_completed(futures):
            result = future.result()
            all_params.extend(result)

    # quit drivers after scraping is complete
    graceful_shutdown(None, None)

    return all_params

# main scraping logic
if __name__ == "__main__":
    # enable logging to track progress
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

    # load the full dataset (all flats)
    df = pd.read_csv('apartments_Praha.csv')

    # configure number of browser windows (based on your system capacity)
    num_workers = 4  # adjust based on system capacity

    # scrape pages with the specified number of browser windows (manual consent, visible mode)
    try:
        scraped_params = scrape_pages_with_fixed_drivers(df, num_workers=num_workers)

        # convert scraped data to DataFrame
        params_df = pd.DataFrame(scraped_params)

        # merge the original DataFrame with the scraped data on the 'Link' column
        merged_df = pd.merge(df, params_df, on='Link', how='left')

        # save the updated DataFrame to a CSV file
        merged_df.to_csv('updated_apartments_Praha.csv', index=False)

        logging.info("Scraping completed and data saved to 'updated_apartments_Praha.csv'")
    except KeyboardInterrupt:
        graceful_shutdown(None, None)